# Brain MRI Tumor Classifier: training notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AakashRanjan1811/Brain-MRI-Tumor-Classifier/blob/main/notebooks/train_colab.ipynb)

Runs the full pipeline from this repository on a GPU:

1. Downloads the Kaggle *Brain Tumor MRI* dataset.
2. Removes duplicate / leaked images (perceptual hashing) and builds a stratified train / validation split. The official test set is used **only once**, for the final evaluation.
3. Trains the **baseline CNN** (the original architecture) and **EfficientNetB0** (ImageNet transfer learning, two-phase fine-tuning).
4. Evaluates both on the test set and produces training curves, confusion matrices and Grad-CAM heatmaps.

**Runtime → Change runtime type → T4 GPU** before running. Takes roughly 25–40 minutes.

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone --depth 1 https://github.com/AakashRanjan1811/Brain-MRI-Tumor-Classifier.git
%cd Brain-MRI-Tumor-Classifier
!pip install -q kagglehub

In [ ]:
import kagglehub
DATA_DIR = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print(DATA_DIR)

In [ ]:
!python -m src.run --data-dir "$DATA_DIR" --out results

In [ ]:
from IPython.display import Image, display
for f in ["training_curves.png", "confusion_matrices.png", "gradcam.jpg"]:
    display(Image(f"results/figures/{f}"))

Compact export of the results (metrics JSON + Grad-CAM figure as base64), used to update the repository README.

In [ ]:
import base64, json
print("===METRICS_JSON_BEGIN===")
print(json.dumps(json.load(open("results/metrics.json")), separators=(",", ":")))
print("===METRICS_JSON_END===")
print("===GRADCAM_B64_BEGIN===")
print(base64.b64encode(open("results/figures/gradcam.jpg", "rb").read()).decode())
print("===GRADCAM_B64_END===")

Optional: download the trained models.

In [ ]:
!zip -qr trained_models.zip results/models/*.keras
from google.colab import files
files.download("trained_models.zip")